In [4]:
import json
import os
import dotenv
from openai import OpenAI
dotenv.load_dotenv()

def get_weather(location:str)->str:
    return f"Current weather in {location} is sunny"


tools_by_name = {"get_weather":get_weather}
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA"
                    }
                },
                "required": ["location"]
            }
        }
    }
]

client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"),base_url="https://api.deepseek.com")

def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
        tools=tools,
        extra_body={"thinking":{"type":"disabled"}}
    )
    return response.choices[0].message

def execute_tool(message,messages):
    tool_call = message.tool_calls[0]
    tool_id = tool_call.id
    tool_name = tool_call.function.name
    tool_args = json.loads(tool_call.function.arguments)
    print(f"调用工具：{tool_name},参数：{tool_args}")
    result = tools_by_name[tool_name](**tool_args)
    print(f"工具结果：{result}")
    messages.append(message)
    messages.append({
        "role":"tool",
        "tool_call_id":tool_id,
        "content":result
    })
    return messages



In [5]:
#调用LLM
messages = [{"role":"user","content":"杭州今天天气如何？"}]
print(f"User>{messages[0]['content']}")
message = send_messages(messages)
print(f"Tool>{message.tool_calls[0]}")


User>杭州今天天气如何？
Tool>ChatCompletionMessageFunctionToolCall(id='call_00_1l7s11OiwXRVb8vnhJop2700', function=Function(arguments='{"location": "杭州, 浙江省"}', name='get_weather'), type='function', index=0)


In [6]:
messages=execute_tool(message,messages)
message = send_messages(messages)
print(f"AI> {message.content}")

调用工具：get_weather,参数：{'location': '杭州, 浙江省'}
工具结果：Current weather in 杭州, 浙江省 is sunny
AI> 杭州今天天气是晴天（sunny）。☀️

如果你需要更详细的信息（如气温、湿度、风力等），我可以再帮你查询。
